# 타슈 수요예측 — 서빙용 예측 함수

`predict(date, hour, station_id)` → `{'rent': 대여예측, 'return': 반납예측}`

- 달력 피처(dow, is_weekend, month, is_holiday)는 date에서 자동 파생
- 날씨 4개(temp, humid, wind, precip)는 **기상청 단기예보 API** 에서 조회
- 미세먼지 3개(pm10, pm25, ozone)는 **에어코리아 실시간 API** (성남동1) 최신값 사용
- `weather=` 인자로 직접 값을 넘기면 API 호출 없이 예측 (테스트/과거날짜용)

**한계**: 기상청 단기예보는 +3일까지만 제공 → 그보다 먼 미래는 예보값이 없어 예측 불가.
미세먼지는 미래 '수치' 예보가 없어서 현재 관측값을 그대로 쓰는 근사(persistence)임.

API 키: https://www.data.go.kr 에서
- 기상청_단기예보 조회서비스
- 한국환경공단_에어코리아_대기오염정보
두 개 활용신청 후 발급받은 일반 인증키(Decoding)를 아래 SERVICE_KEY에 입력.

In [ ]:
import pandas as pd
import joblib
import requests
from datetime import datetime, timedelta

SERVICE_KEY = '여기에_공공데이터포털_인증키'   # 기상청/에어코리아 공용 (data.go.kr 키는 계정당 하나)

# ── 모델 로드 ─────────────────────────────
model_rent   = joblib.load('../models/model_rent_v1.pkl')
model_return = joblib.load('../models/model_return_v1.pkl')

# 학습 때 category 인코딩과 동일해야 함 (train.csv sorted unique — 순서 바꾸면 안 됨!)
STATIONS = ['ST0056', 'ST0065', 'ST0082', 'ST0107', 'ST0342', 'ST0627', 'ST0628',
            'ST0639', 'ST0641', 'ST0642', 'ST0685', 'ST0720', 'ST0934', 'ST1067',
            'ST1089', 'ST1302', 'ST1303', 'ST1330', 'ST1334']

X_cols = ['station_id', 'hour', 'dow', 'is_weekend', 'month', 'is_holiday',
          'temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']

## 1. 달력 파생 피처

전처리 때는 공휴일을 하드코딩했지만, 서빙에선 임의 미래 날짜가 오므로 `holidays` 라이브러리 사용.
(`pip install holidays`) 단, **임시공휴일은 라이브러리가 모름** → 발표되면 EXTRA_HOLIDAYS에 수동 추가.

In [ ]:
import holidays as hl

kr_holidays = hl.KR(years=range(2025, 2031))

# 임시공휴일 등 라이브러리에 없는 날 (전처리 하드코딩 리스트와 정합성 유지)
EXTRA_HOLIDAYS = {'2025-01-27', '2026-01-26'}   # 예: 설연휴 임시공휴일 (2025-06-03 대선일 등 필요 시 추가)

def calendar_feats(date):
    d = pd.Timestamp(date)
    return {
        'dow':        d.dayofweek,                  # 0=월 ~ 6=일
        'is_weekend': int(d.dayofweek >= 5),
        'month':      d.month,
        'is_holiday': int(d.date() in kr_holidays or d.strftime('%Y-%m-%d') in EXTRA_HOLIDAYS),
    }

calendar_feats('2026-07-20')   # 월요일, 평일

## 2. 기상청 단기예보 → temp, humid, wind, precip

- 대전 동구/중구 (대전역 일대) 격자: **nx=68, ny=100** ← 기상청 격자변환표로 한번 검증할 것
- 발표시각: 02/05/08/11/14/17/20/23시 (제공은 각 +10분 뒤) → 현재 시각 기준 가장 최근 발표분 사용
- 매핑: TMP→temp(℃), REH→humid(%), WSD→wind(m/s), PCP→precip(mm)

In [ ]:
NX, NY = 68, 100   # 대전 동구 부근
KMA_URL = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst'
BASE_HOURS = [2, 5, 8, 11, 14, 17, 20, 23]

def _latest_base():
    """현재 시각 기준 가장 최근 발표 base_date, base_time"""
    now = datetime.now() - timedelta(minutes=30)   # 발표 직후 지연 여유
    for _ in range(2):
        cands = [h for h in BASE_HOURS if h <= now.hour]
        if cands:
            return now.strftime('%Y%m%d'), f'{max(cands):02d}00'
        now -= timedelta(hours=3)   # 자정 직후면 전날 23시 발표분
    return now.strftime('%Y%m%d'), '2300'

def _parse_pcp(v):
    """PCP는 문자열: '강수없음', '1mm 미만', '5.0mm', '30.0~50.0mm' 등"""
    if v in ('강수없음', None, '-', ''): return 0.0
    if '미만' in v: return 0.5
    v = v.replace('mm', '').split('~')[-1]
    try: return float(v)
    except ValueError: return 0.0

def fetch_weather(date, hour):
    """(date, hour)의 예보값 → {'temp','humid','wind','precip'} / 예보 범위 밖이면 ValueError"""
    base_date, base_time = _latest_base()
    r = requests.get(KMA_URL, params={
        'serviceKey': SERVICE_KEY, 'dataType': 'JSON',
        'base_date': base_date, 'base_time': base_time,
        'nx': NX, 'ny': NY, 'numOfRows': 1000, 'pageNo': 1,
    }, timeout=10)
    items = r.json()['response']['body']['items']['item']

    want_date = pd.Timestamp(date).strftime('%Y%m%d')
    want_time = f'{hour:02d}00'
    vals = {i['category']: i['fcstValue'] for i in items
            if i['fcstDate'] == want_date and i['fcstTime'] == want_time}
    if 'TMP' not in vals:
        raise ValueError(f'{date} {hour}시는 예보 범위 밖 (단기예보는 +3일까지)')
    return {
        'temp':   float(vals['TMP']),
        'humid':  float(vals['REH']),
        'wind':   float(vals['WSD']),
        'precip': _parse_pcp(vals.get('PCP')),
    }

## 3. 에어코리아 실시간 → pm10, pm25, ozone

미래 시각의 '수치' 예보 API는 없음(등급 예보만 있음).
→ 성남동1 측정소 **최신 관측값을 그대로 사용** (미세먼지는 몇 시간 단위론 크게 안 변한다는 근사).
학습 데이터와 동일 측정소·동일 단위(㎍/㎥, ppm)라 스케일 문제 없음.

In [ ]:
AIR_URL = 'http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty'

def fetch_air():
    """성남동1 최신 관측 → {'pm10','pm25','ozone'}"""
    r = requests.get(AIR_URL, params={
        'serviceKey': SERVICE_KEY, 'returnType': 'json',
        'stationName': '성남동1', 'dataTerm': 'DAILY', 'numOfRows': 24, 'pageNo': 1, 'ver': '1.3',
    }, timeout=10)
    items = r.json()['response']['body']['items']

    def latest(key):
        for it in items:   # 최신순 정렬 — 통신장애('-') 건너뛰고 첫 유효값
            v = it.get(key)
            if v not in (None, '-', ''): return float(v)
        return None
    return {'pm10': latest('pm10Value'), 'pm25': latest('pm25Value'), 'ozone': latest('o3Value')}

## 4. 통합 predict 함수

In [ ]:
def predict(date, hour, station_id, weather=None):
    """
    date: 'YYYY-MM-DD', hour: 0~23, station_id: 'ST0065' 등

    weather: None이면 기상청+에어코리아 API 호출 (미래/현재용).
             dict로 7개 값을 직접 주면 API 없이 예측 (과거 재현·테스트용)
             예: {'temp':-2.2,'humid':59,'wind':0.9,'precip':0,'pm10':20,'pm25':13,'ozone':0.006}

    반환: {'rent': float, 'return': float}
    """
    if station_id not in STATIONS:
        raise ValueError(f'모르는 스테이션: {station_id}')

    if weather is None:
        weather = {**fetch_weather(date, hour), **fetch_air()}

    row = {'station_id': station_id, 'hour': hour, **calendar_feats(date), **weather}
    X = pd.DataFrame([row])[X_cols]
    # 학습 때와 동일한 category 인코딩 (categories 순서까지 일치해야 함)
    X['station_id'] = pd.Categorical(X['station_id'], categories=STATIONS)

    return {
        'rent':   round(float(model_rent.predict(X)[0]), 2),
        'return': round(float(model_return.predict(X)[0]), 2),
    }

## 5. 테스트

### 5-1. 과거 데이터로 정합성 확인 (API 불필요)
test.csv의 실제 날씨값을 weather로 넘겨서 추론테스트 노트북과 같은 결과가 나오는지 확인.

In [ ]:
test = pd.read_csv('../preprocessing/test.csv')
sample = test[(test['date'] == '2026-02-10') & (test['station_id'] == 'ST0065') & (test['hour'] == 17)].iloc[0]

w = {k: sample[k] for k in ['temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']}
pred = predict('2026-02-10', 17, 'ST0065', weather=w)
print(f'예측: {pred}')
print(f"실제: rent={sample['rent_cnt']}, return={sample['return_cnt']}")

### 5-2. 실제 API로 미래 예측 (SERVICE_KEY 필요)

In [ ]:
# SERVICE_KEY 입력 후 실행
tomorrow = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
print(f'{tomorrow} 17시 대전역(ST0065):', predict(tomorrow, 17, 'ST0065'))

In [ ]:
# 하루 24시간 전체 예측 (대시보드용 데이터 형태)
def predict_day(date, station_id):
    air = fetch_air()   # 미세먼지는 하루 한 번만 조회
    rows = []
    for h in range(24):
        try:
            w = {**fetch_weather(date, h), **air}
        except ValueError:
            continue   # 예보 범위 밖 시간대는 건너뜀
        rows.append({'hour': h, **predict(date, h, station_id, weather=w)})
    return pd.DataFrame(rows)

# predict_day(tomorrow, 'ST0065')